# Data Cleaning & Visualization

This notebook walks through inspection, cleaning, outlier treatment, exploratory analysis, and visualization of the retail sales dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw_sales_data.csv')
df.head()

In [ ]:
print(df.info())
print('Missing values:\n', df.isna().sum())
print('Duplicate transaction IDs:', df['transaction_id'].duplicated().sum())

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'])
df['category'] = df['category'].astype('string').str.strip().str.title()
df['region'] = df['region'].astype('string').str.strip().str.title()
for col in ['units_sold', 'unit_price', 'discount', 'revenue']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['units_sold'] = df['units_sold'].fillna(df['units_sold'].median())
df['discount'] = df['discount'].fillna(df['discount'].median())
df = df.drop_duplicates(subset=['transaction_id'])
q1, q3 = df['units_sold'].quantile([0.25, 0.75])
iqr = q3 - q1
df['units_sold'] = df['units_sold'].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
df['revenue'] = df['units_sold'] * df['unit_price'] * (1 - df['discount'])
df['month'] = df['date'].dt.to_period('M').astype(str)
df.to_csv('../data/cleaned_sales_data.csv', index=False)
df.head()

In [ ]:
sns.set_theme(style='whitegrid')
monthly = df.groupby('month', as_index=False)['revenue'].sum()
sns.lineplot(data=monthly, x='month', y='revenue', marker='o')
plt.title('Monthly Revenue Trend')
plt.xticks(rotation=45)
plt.show()

In [ ]:
category = df.groupby('category', as_index=False)['revenue'].sum().sort_values('revenue', ascending=False)
sns.barplot(data=category, x='category', y='revenue')
plt.title('Revenue by Product Category')
plt.xticks(rotation=20)
plt.show()

## Key Takeaway

The analysis demonstrates how preprocessing decisions directly affect downstream visualizations. Always inspect data quality before drawing conclusions from aggregated metrics.